# Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from echoft.inference import run_inference, get_samples
from echoft.postprocess import (
    reconstruct_lightcurve_samples,
    summarise_posterior,
    to_arviz,
)
from echoft.model import evaluate_echo_model

import arviz as az

# Data Ingestion

In [ ]:
data = np.load("../data/synthetic_lightcurves.npz")

time = data["time"]

flux_dict = {
    "xray": data["flux_xray"],
    "uv": data["flux_uv"],
    "optical": data["flux_optical"],
}

sigma_dict = {
    "xray": data["sigma_xray"],
    "uv": data["sigma_uv"],
    "optical": data["sigma_optical"],
}

wavelengths = {
    "uv": 1500,
    "optical": 5000,
}

# Run

In [ ]:
mcmc = run_inference(time, flux_dict, sigma_dict)
samples = get_samples(mcmc)

# Convergence Diagnostics

In [ ]:
idata = to_arviz(mcmc)

az.plot_trace(idata)
plt.show()

# Posterior Dstributions

In [ ]:
az.plot_posterior(idata)
plt.show()

# Light curve fits

In [ ]:
def reconstruct_multiband(time, flux_dict, sigma_dict, samples, n_draws=100):
    idx = np.random.choice(len(samples["M_BH"]), n_draws)

    models = {band: [] for band in flux_dict}

    for i in idx:
        params = (
            samples["M_BH"][i],
            samples["acc_rate"][i],
            samples["incl"][i],
        )

        model_dict, _ = evaluate_echo_model(
            time, flux_dict, sigma_dict, params, frequencies
        )

        for band in flux_dict:
            models[band].append(model_dict[band])

    for band in models:
        models[band] = np.array(models[band])

    return models

In [ ]:
models = reconstruct_multiband(time, flux_dict, sigma_dict, samples)

for band in flux_dict:
    median = np.median(models[band], axis=0)
    lo = np.percentile(models[band], 16, axis=0)
    hi = np.percentile(models[band], 84, axis=0)

    plt.figure()
    plt.errorbar(time, flux_dict[band], yerr=sigma_dict[band], fmt='.')
    plt.plot(time, median)
    plt.fill_between(time, lo, hi, alpha=0.3)
    plt.title(band)
    plt.show()